# Chapter 24
## Synchronization by Fast Recurrent Excitation
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter24.ipynb)

## About this chapter

Fast recurrent excitation can bring initially dispersed RTM neurons into
a common firing rhythm: $I_{{\rm syn},i}=g_{\rm syn}\sum_{j\ne i}s_j(v_{\rm syn}-v_i)$.
Every network below is initialized by finding the single-cell RTM limit
cycle (`rtm_init`) and spreading each neuron's phase around it.
`simulate_rtm_sync` starts every neuron at the same phase (no synapse) as
the synchronous reference; `simulate_rtm_splay` spreads phases evenly
with no synapse, the asynchronous reference. `simulate_rtm_e_to_e_network`
couples all neurons through a shared mean-field synapse, called once for
a short run and once for a much longer run (`simulate_rtm_e_to_e_network_2`
below) to watch the splay state contract toward synchrony -- the long
run's inner loop is JIT-compiled with numba, since the uncompiled 10^6-step
sweep took minutes. `simulate_rtm_e_to_e_heterogeneous` uses random
per-neuron drive and per-pair coupling strength instead of identical
neurons and a shared synapse. `simulate_rtm_two_cell_network` makes the
phase interaction visible in a reciprocally excitatory two-cell network,
including a deliberate tiny perturbation partway through to probe
stability of the synchronous state.

See [`README.md`](chapter24.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from numba import njit
from mnd.core import draw_arrow

## RTM Limit Cycle and Release Time Constant (shared by every example below)

`rtm_init` finds the single-cell RTM limit cycle at a given drive (plain
Heun integration until the 5th spike) and interpolates $(v,h,n)$ at each
requested phase -- this is how every network below builds phase-spread
initial conditions.

In [ ]:
@njit
def _rtm_alpha_h(v):
    return 0.128 * np.exp(-(v + 50.0) / 18.0)


@njit
def _rtm_alpha_m(v):
    return 0.32 * (v + 54.0) / (1.0 - np.exp(-(v + 54.0) / 4.0))


@njit
def _rtm_alpha_n(v):
    return 0.032 * (v + 52.0) / (1.0 - np.exp(-(v + 52.0) / 5.0))


@njit
def _rtm_beta_h(v):
    return 4.0 / (1.0 + np.exp(-(v + 27.0) / 5.0))


@njit
def _rtm_beta_m(v):
    return 0.28 * (v + 27.0) / (np.exp((v + 27.0) / 5.0) - 1.0)


@njit
def _rtm_beta_n(v):
    return 0.5 * np.exp(-(v + 57.0) / 40.0)


@njit
def _rtm_m_inf(v):
    am, bm = _rtm_alpha_m(v), _rtm_beta_m(v)
    return am / (am + bm)


@njit
def _rtm_h_inf(v):
    ah, bh = _rtm_alpha_h(v), _rtm_beta_h(v)
    return ah / (ah + bh)


@njit
def _rtm_n_inf(v):
    an, bn = _rtm_alpha_n(v), _rtm_beta_n(v)
    return an / (an + bn)


@njit
def tau_peak_function(tau_d, tau_r, tau_d_q):
    """Time (from a delta-function pulse of transmitter release) at which
    the synaptic gate s peaks."""
    dt = 0.01
    dt05 = dt / 2
    s, t = 0.0, 0.0
    s_inc = math.exp(-t / tau_d_q) * (1 - s) / tau_r - s * tau_d
    while s_inc > 0:
        t_old, s_inc_old = t, s_inc
        s_tmp = s + dt05 * s_inc
        s_inc_tmp = math.exp(-(t + dt05) / tau_d_q) * (1 - s_tmp) / tau_r - s_tmp / tau_d
        s = s + dt * s_inc_tmp
        t = t + dt
        s_inc = math.exp(-t / tau_d_q) * (1 - s) / tau_r - s / tau_d
    return (t_old * (-s_inc) + t * s_inc_old) / (s_inc_old - s_inc)


@njit
def tau_d_q_function(tau_d, tau_r, tau_hat):
    """Release time constant tau_d_q so that tau_peak_function reproduces
    the prescribed tau_hat (bisection, since there's no closed form)."""
    tau_d_q_left = 1.0
    while tau_peak_function(tau_d, tau_r, tau_d_q_left) > tau_hat:
        tau_d_q_left /= 2
    tau_d_q_right = tau_r
    while tau_peak_function(tau_d, tau_r, tau_d_q_right) < tau_hat:
        tau_d_q_right *= 2
    while tau_d_q_right - tau_d_q_left > 1e-12:
        tau_d_q_mid = (tau_d_q_left + tau_d_q_right) / 2
        if tau_peak_function(tau_d, tau_r, tau_d_q_mid) <= tau_hat:
            tau_d_q_left = tau_d_q_mid
        else:
            tau_d_q_right = tau_d_q_mid
    return (tau_d_q_left + tau_d_q_right) / 2


@njit
def rtm_init(i_ext, phi_vec, c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
             v_k=-100.0, v_na=50.0, v_l=-67.0, t_final=5000.0, dt=0.001, v0=-70.0):
    """Find the RTM limit cycle at i_ext (plain-float Heun integration
    until the 5th spike), then interpolate (v, h, n) at each phase in
    phi_vec (fraction of the last full period, measured from the 4th
    spike). Returns (len(phi_vec), 3) array of initial conditions and
    the period T (np.inf if i_ext is subthreshold)."""
    dt05 = dt / 2

    v = [v0]
    m = [_rtm_m_inf(v0)]
    h = [_rtm_h_inf(v0)]
    n = [_rtm_n_inf(v0)]
    t_spikes = []

    k = 0
    t = 0.0
    while len(t_spikes) < 5 and t < t_final:
        vk, mk, hk, nk = v[k], m[k], h[k], n[k]
        v_inc = (g_k * math.pow(nk, 4.0) * (v_k - vk) + g_na * math.pow(mk, 3.0) * hk * (v_na - vk)
                 + g_l * (v_l - vk) + i_ext) / c
        h_inc = _rtm_alpha_h(vk) * (1 - hk) - _rtm_beta_h(vk) * hk
        n_inc = _rtm_alpha_n(vk) * (1 - nk) - _rtm_beta_n(vk) * nk

        v_tmp = vk + dt05 * v_inc
        m_tmp = _rtm_m_inf(v_tmp)
        h_tmp = hk + dt05 * h_inc
        n_tmp = nk + dt05 * n_inc

        v_inc = (g_k * math.pow(n_tmp, 4.0) * (v_k - v_tmp) + g_na * math.pow(m_tmp, 3.0) * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
        n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp

        v.append(vk + dt * v_inc)
        h.append(hk + dt * h_inc)
        n.append(nk + dt * n_inc)
        m.append(_rtm_m_inf(v[-1]))

        if vk >= -20 and v[-1] < -20:
            t_spike = ((k) * dt * (-20 - v[-1]) + (k + 1) * dt * (20 + vk)) / (vk - v[-1])
            t_spikes.append(t_spike)
        t = (k + 1) * dt
        k += 1

    num = len(phi_vec)
    if len(t_spikes) < 5:
        v_last, h_last, n_last = v[k], h[k], n[k]
        out = np.zeros((num, 3))
        for i in range(num):
            out[i, 0], out[i, 1], out[i, 2] = v_last, h_last, n_last
        return out, np.inf

    T = t_spikes[4] - t_spikes[3]
    out = np.zeros((num, 3))
    for i, phi0 in enumerate(phi_vec):
        t0 = phi0 * T + t_spikes[3]
        kk = int(t0 / dt)
        frac_hi = (t0 - kk * dt) / dt
        frac_lo = ((kk + 1) * dt - t0) / dt
        out[i, 0] = v[kk + 1] * frac_hi + v[kk] * frac_lo
        out[i, 1] = h[kk + 1] * frac_hi + h[kk] * frac_lo
        out[i, 2] = n[kk + 1] * frac_hi + n[kk] * frac_lo
    return out, T


@njit
def spike_times_and_indices(v_old, v, k, dt):
    """Threshold crossings (-20mV) between consecutive states of an
    (N,)-shaped population, as (times, 1-indexed neuron numbers)."""
    ind = np.where((v_old >= -20) & (v < -20))[0]
    if len(ind) == 0:
        return np.empty(0), np.empty(0, dtype=np.int64)
    t_new = ((k - 1) * dt * (-v[ind] - 20) + k * dt * (20 + v_old[ind])) / (v_old[ind] - v[ind])
    return t_new, ind + 1

## Synchronous Reference

Every neuron starts at the same state with no synaptic coupling -- the
baseline synchronous raster to compare every other example against.

In [ ]:
@njit
def simulate_rtm_sync(N=30, c=1.0, g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0,
                       i_ext=0.3, t_final=200.0, dt=0.01, v0=-70.0):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = v0 * np.ones(N)
    m = _rtm_m_inf(v)
    h = _rtm_h_inf(v)
    n = _rtm_n_inf(v)

    t_spikes, i_spikes = [], []
    for k in range(1, m_steps + 1):
        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v) + i_ext) / c
        n_inc = _rtm_alpha_n(v) * (1 - n) - _rtm_beta_n(v) * n
        h_inc = _rtm_alpha_h(v) * (1 - h) - _rtm_beta_h(v) * h

        v_tmp = v + dt05 * v_inc
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        m_tmp = _rtm_m_inf(v_tmp)

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
        n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp

        v_old = v
        v = v + dt * v_inc
        h = h + dt * h_inc
        n = n + dt * n_inc
        m = _rtm_m_inf(v)

        ts, ii = spike_times_and_indices(v_old, v, k, dt)
        t_spikes.extend(ts)
        i_spikes.extend(ii)

    return np.array(t_spikes), np.array(i_spikes)


def plot_rtm_raster(t_spikes, i_spikes, N=30, t_final=200.0, xlim=None):
    plt.figure(figsize=(7, 4))
    plt.plot(t_spikes, i_spikes, '.r', markersize=10)
    plt.xlim(xlim if xlim else (0, t_final))
    plt.ylim(0, N + 1)
    plt.xlabel('$t$ [ms]')
    plt.ylabel('neuron #')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_rtm_raster(*simulate_rtm_sync())

## Splay-State Reference

Neurons start with phases spread evenly around the limit cycle, again
with no synaptic coupling -- the asynchronous reference raster.

In [ ]:
@njit
def simulate_rtm_splay(N=30, c=1.0, g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0,
                        i_ext=0.3, t_final=200.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    phi_vec = np.arange(N - 1.0, -1.0, -1.0) / N + 1.0 / (2 * N)
    initial_vector, T = rtm_init(i_ext, phi_vec)

    v = initial_vector[:, 0].copy()
    h = initial_vector[:, 1].copy()
    n = initial_vector[:, 2].copy()
    m = _rtm_m_inf(v)

    t_spikes, i_spikes = [], []
    for k in range(1, m_steps + 1):
        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v) + i_ext) / c
        n_inc = _rtm_alpha_n(v) * (1 - n) - _rtm_beta_n(v) * n
        h_inc = _rtm_alpha_h(v) * (1 - h) - _rtm_beta_h(v) * h

        v_tmp = v + dt05 * v_inc
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        m_tmp = _rtm_m_inf(v_tmp)

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
        n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp

        v_old = v
        v = v + dt * v_inc
        h = h + dt * h_inc
        n = n + dt * n_inc
        m = _rtm_m_inf(v)

        ts, ii = spike_times_and_indices(v_old, v, k, dt)
        t_spikes.extend(ts)
        i_spikes.extend(ii)

    return np.array(t_spikes), np.array(i_spikes)

In [ ]:
plot_rtm_raster(*simulate_rtm_splay())

## Recurrent E-to-E Network

Neurons couple through a shared mean-field synapse,
$I_{{\rm syn},i}=g_{\rm syn}\sum_{j\ne i}s_j(v_{\rm syn}-v_i)$ with
$v_{\rm syn}=0$ folded into the linear $-g_{\rm syn}(\sum_j s_j - s_i)v_i$
form used below, starting from the splay state. A short run
(`simulate_rtm_e_to_e_network`) shows the early contraction toward
synchrony; a much longer run
(`simulate_rtm_e_to_e_network_2`, numba-jitted) settles into the
synchronous rhythm and reports its frequency.

In [ ]:
@njit
def _rtm_meanfield_network_loop(v, m, h, n, N, g_syn, dt, dt05, m_steps,
                                 c, g_k, g_na, g_l, v_k, v_na, v_l, i_ext, tau_r, tau_d, tau_dq):
    q = np.zeros(N)
    s = np.zeros(N)
    t_spikes = np.empty(m_steps * N, dtype=np.float64)
    i_spikes = np.empty(m_steps * N, dtype=np.int64)
    n_spikes = 0

    for k in range(1, m_steps + 1):
        s_sum = s.sum()
        v_old = v.copy()
        v_tmp = np.empty(N); m_tmp = np.empty(N); h_tmp = np.empty(N)
        n_tmp = np.empty(N); q_tmp = np.empty(N); s_tmp = np.empty(N)

        for i in range(N):
            alpha_h_v = 0.128 * math.exp(-(v[i] + 50) / 18)
            alpha_n_v = 0.032 * (v[i] + 52) / (1 - math.exp(-(v[i] + 52) / 5))
            beta_h_v = 4.0 / (1 + math.exp(-(v[i] + 27) / 5))
            beta_n_v = 0.5 * math.exp(-(v[i] + 57) / 40)

            v_inc = (g_k * math.pow(n[i], 4.0) * (v_k - v[i]) + g_na * math.pow(m[i], 3.0) * h[i] * (v_na - v[i])
                     + g_l * (v_l - v[i]) - g_syn * (s_sum - s[i]) * v[i] + i_ext) / c
            h_inc = alpha_h_v * (1 - h[i]) - beta_h_v * h[i]
            n_inc = alpha_n_v * (1 - n[i]) - beta_n_v * n[i]
            q_inc = 5 * (1 + math.tanh(v[i] / 10)) * (1 - q[i]) - q[i] / tau_dq
            s_inc = q[i] * (1 - s[i]) / tau_r - s[i] / tau_d

            v_tmp[i] = v[i] + dt05 * v_inc
            h_tmp[i] = h[i] + dt05 * h_inc
            n_tmp[i] = n[i] + dt05 * n_inc
            q_tmp[i] = q[i] + dt05 * q_inc
            s_tmp[i] = s[i] + dt05 * s_inc
            m_tmp[i] = _rtm_m_inf(v_tmp[i])

        s_tmp_sum = s_tmp.sum()
        for i in range(N):
            alpha_h_vtmp = 0.128 * math.exp(-(v_tmp[i] + 50) / 18)
            alpha_n_vtmp = 0.032 * (v_tmp[i] + 52) / (1 - math.exp(-(v_tmp[i] + 52) / 5))
            beta_h_vtmp = 4.0 / (1 + math.exp(-(v_tmp[i] + 27) / 5))
            beta_n_vtmp = 0.5 * math.exp(-(v_tmp[i] + 57) / 40)

            v_inc = (g_k * math.pow(n_tmp[i], 4.0) * (v_k - v_tmp[i])
                     + g_na * math.pow(m_tmp[i], 3.0) * h_tmp[i] * (v_na - v_tmp[i])
                     + g_l * (v_l - v_tmp[i]) - g_syn * (s_tmp_sum * v_tmp[i] - s_tmp[i] * v_tmp[i]) + i_ext) / c
            h_inc = alpha_h_vtmp * (1 - h_tmp[i]) - beta_h_vtmp * h_tmp[i]
            n_inc = alpha_n_vtmp * (1 - n_tmp[i]) - beta_n_vtmp * n_tmp[i]
            q_inc = 5 * (1 + math.tanh(v_tmp[i] / 10)) * (1 - q_tmp[i]) - q_tmp[i] / tau_dq
            s_inc = q_tmp[i] * (1 - s_tmp[i]) / tau_r - s_tmp[i] / tau_d

            v[i] = v[i] + dt * v_inc
            h[i] = h[i] + dt * h_inc
            n[i] = n[i] + dt * n_inc
            q[i] = q[i] + dt * q_inc
            s[i] = s[i] + dt * s_inc
            m[i] = _rtm_m_inf(v[i])

            if v_old[i] >= -20 and v[i] < -20:
                t_new = ((k - 1) * dt * (-v[i] - 20) + k * dt * (20 + v_old[i])) / (-v[i] + v_old[i])
                t_spikes[n_spikes] = t_new
                i_spikes[n_spikes] = i + 1
                n_spikes += 1

    return t_spikes[:n_spikes], i_spikes[:n_spikes]


def simulate_rtm_e_to_e_network(N=30, c=1.0, g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0,
                                 i_ext=0.30, g_syn=0.0075, tau_r=0.5, tau_peak=0.5, tau_d=2.0,
                                 t_final=200.0, dt=0.01):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq = tau_d_q_function(tau_d, tau_r, tau_peak)

    phi_vec = np.arange(N - 1.0, -1.0, -1.0) / N + 1.0 / (2 * N)
    initial_vector, T = rtm_init(i_ext, phi_vec)
    v = initial_vector[:, 0].copy()
    h = initial_vector[:, 1].copy()
    n = initial_vector[:, 2].copy()
    m = _rtm_m_inf(v)

    t_spikes, i_spikes = _rtm_meanfield_network_loop(
        v, m, h, n, N, g_syn, dt, dt05, m_steps, c, g_k, g_na, g_l, v_k, v_na, v_l, i_ext, tau_r, tau_d, tau_dq)
    return t_spikes, i_spikes


def simulate_rtm_e_to_e_network_2(**kwargs):
    kwargs.setdefault("t_final", 10000.0)
    return simulate_rtm_e_to_e_network(**kwargs)

In [ ]:
plot_rtm_raster(*simulate_rtm_e_to_e_network())

In [ ]:
# Numba-compiled, so the 10^6-step sweep now takes seconds instead of minutes.
t_spikes, i_spikes = simulate_rtm_e_to_e_network_2()
ind1 = np.where(i_spikes == 1)[0]
frequency = 1000 / (t_spikes[ind1[-1]] - t_spikes[ind1[-2]])
print(f"frequency = {frequency}")
plot_rtm_raster(t_spikes, i_spikes, t_final=10000.0, xlim=(10000.0 - 200, 10000.0))

## Heterogeneous Recurrent Network

Each neuron gets its own random drive and each ordered pair its own
random coupling strength, instead of identical neurons and a shared
synapse -- tests how robust the synchronization mechanism is to
heterogeneity.

In [ ]:
@njit
def _rtm_heterogeneous_loop(v, h, n, m, q, s, N, g_syn_T, i_ext, dt, dt05, m_steps,
                             c, g_k, g_na, g_l, v_k, v_na, v_l, tau_r, tau_d, tau_dq):
    t_spikes, i_spikes = [], []
    for k in range(1, m_steps + 1):
        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v)
                 + g_l * (v_l - v) - (g_syn_T @ s) * v + i_ext) / c
        h_inc = _rtm_alpha_h(v) * (1 - h) - _rtm_beta_h(v) * h
        n_inc = _rtm_alpha_n(v) * (1 - n) - _rtm_beta_n(v) * n
        q_inc = 5 * (1 + np.tanh(v / 10)) * (1 - q) - q / tau_dq
        s_inc = q * (1 - s) / tau_r - s / tau_d

        v_tmp = v + dt05 * v_inc
        m_tmp = _rtm_m_inf(v_tmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        q_tmp = q + dt05 * q_inc
        s_tmp = s + dt05 * s_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) - (g_syn_T @ s_tmp) * v_tmp + i_ext) / c
        h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
        n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp
        q_inc = 5 * (1 + np.tanh(v_tmp / 10)) * (1 - q_tmp) - q_tmp / tau_dq
        s_inc = q_tmp * (1 - s_tmp) / tau_r - s_tmp / tau_d

        v_old = v.copy()
        v = v + dt * v_inc
        m = _rtm_m_inf(v)
        h = h + dt * h_inc
        n = n + dt * n_inc
        q = q + dt * q_inc
        s = s + dt * s_inc

        ts, ii = spike_times_and_indices(v_old, v, k, dt)
        t_spikes.extend(ts)
        i_spikes.extend(ii)

    return t_spikes, i_spikes


def simulate_rtm_e_to_e_heterogeneous(N=30, c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                       v_k=-100.0, v_na=50.0, v_l=-67.0,
                                       tau_r=0.5, tau_peak=0.5, tau_d=2.0,
                                       t_final=200.0, dt=0.01, seed=63806):
    # matlab's rng(\'default\'); rng(63806) seeds MATLAB\'s own Mersenne-Twister
    # stream, which numpy cannot reproduce bit-for-bit -- this port uses its
    # own numpy seed instead, so the specific realization of random i_ext/
    # g_syn (and hence the exact spike times) differs from matlab\'s, even
    # though the model and algorithm are identical.
    # (RNG setup stays plain Python -- numba nopython mode does not support
    # np.random.Generator / default_rng.)
    rng = np.random.default_rng(seed)
    i_ext = 0.25 + rng.random(N) * 0.1
    g_syn = 0.00625 + rng.random((N, N)) * 0.0025
    np.fill_diagonal(g_syn, 0.0)

    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq = tau_d_q_function(tau_d, tau_r, tau_peak)

    v = np.zeros(N)
    h = np.zeros(N)
    n = np.zeros(N)
    for i in range(N):
        v[i], h[i], n[i] = rtm_init(i_ext[i], np.array([rng.random()]))[0][0]
    m = _rtm_m_inf(v)
    q = np.zeros(N)
    s = np.zeros(N)
    g_syn_T = g_syn.T

    t_spikes, i_spikes = _rtm_heterogeneous_loop(
        v, h, n, m, q, s, N, g_syn_T, i_ext, dt, dt05, m_steps,
        c, g_k, g_na, g_l, v_k, v_na, v_l, tau_r, tau_d, tau_dq)
    return np.array(t_spikes), np.array(i_spikes), i_ext, g_syn

In [ ]:
t_spikes, i_spikes, i_ext, g_syn = simulate_rtm_e_to_e_heterogeneous()
ind1 = np.where(i_spikes == 1)[0]
frequency = 1000 / (t_spikes[ind1[-1]] - t_spikes[ind1[-2]]) if len(ind1) >= 2 else np.nan
print(f"frequency = {frequency}")
plot_rtm_raster(t_spikes, i_spikes, xlim=(200.0 - 200, 200.0))

## Two-Cell Reciprocal Network

Two identical, reciprocally excitatory RTM neurons started in near-perfect
synchrony; a deliberate $10^{-5}$mV perturbation to neuron 1 after its
10th spike probes whether the synchronous state is stable (spike-time
difference returning to ~0) or unstable (difference growing).

In [ ]:
@njit
def simulate_rtm_two_cell_network(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0,
                                   i_ext=0.3, tau_r=0.5, tau_peak=0.5, tau_d=2.0,
                                   g_syn=0.0075 * 29, t_final=1000.0, dt=0.01, perturb_after=10):
    N = 2
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq = tau_d_q_function(tau_d, tau_r, tau_peak)

    initial_vector, T = rtm_init(i_ext, np.array([0.4, 0.4]))
    v = initial_vector[:, 0].copy()
    h = initial_vector[:, 1].copy()
    n = initial_vector[:, 2].copy()
    m = _rtm_m_inf(v)
    q = np.zeros(N)
    s = np.zeros(N)

    t_spikes, i_spikes = [], []
    done = False
    for k in range(1, m_steps + 1):
        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v)
                 + g_l * (v_l - v) - g_syn * (s.sum() - s) * v + i_ext) / c
        h_inc = _rtm_alpha_h(v) * (1 - h) - _rtm_beta_h(v) * h
        n_inc = _rtm_alpha_n(v) * (1 - n) - _rtm_beta_n(v) * n
        # faithful port: this first stage divides by tau_d, not tau_dq
        # (matches the book's original script -- see notebook note above)
        q_inc = 5 * (1 + np.tanh(v / 10)) * (1 - q) - q / tau_d
        s_inc = q * (1 - s) / tau_r - s / tau_d

        v_tmp = v + dt05 * v_inc
        m_tmp = _rtm_m_inf(v_tmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        q_tmp = q + dt05 * q_inc
        s_tmp = s + dt05 * s_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) - g_syn * (s_tmp.sum() * v_tmp - s_tmp * v_tmp) + i_ext) / c
        h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
        n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp
        q_inc = 5 * (1 + np.tanh(v_tmp / 10)) * (1 - q_tmp) - q_tmp / tau_dq
        s_inc = q_tmp * (1 - s_tmp) / tau_r - s_tmp / tau_d

        v_old = v.copy()
        v = v + dt * v_inc
        m = _rtm_m_inf(v)
        h = h + dt * h_inc
        n = n + dt * n_inc
        q = q + dt * q_inc
        s = s + dt * s_inc

        ts, ii = spike_times_and_indices(v_old, v, k, dt)
        if len(ts) > 0:
            t_spikes.extend(ts)
            i_spikes.extend(ii)
            if len(t_spikes) == perturb_after and not done:
                v[0] -= 1e-5
                done = True

    t_spikes = np.array(t_spikes)
    i_spikes = np.array(i_spikes)
    return t_spikes, i_spikes


def plot_rtm_two_cell_network(t_spikes, i_spikes):
    t1 = t_spikes[i_spikes == 1]
    t2 = t_spikes[i_spikes == 2]
    diff = t2 - t1

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(np.arange(1, len(diff) + 1), diff, '.k', markersize=15)
    ax.set_xlabel('spike #')
    ax.set_title('spike time difference [ms]')
    ax.set_xlim(0, 25)
    ax.set_ylim(-2, 2)

    ax.plot([5, 5], [-3, -4], clip_on=False, color='k')
    ax.text(0, -4.5, 'time when $v_1$ is lowered', fontsize=12)
    draw_arrow(ax, (0, 25), (-2, 6), 5, -3, np.array([0., 1.]), epsilon=0.025, width=2, color='k')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_rtm_two_cell_network(*simulate_rtm_two_cell_network())